<h2 style="color:blue" align="center">Training Loss vs Validation Loss</h2>

#### Training vs validation loss

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split
import matplotlib.pyplot as plt

# 1. Split existing training data into Train (80%) and Validation (20%)
# Assuming X_train_tensor and y_train_tensor are already defined
full_dataset = TensorDataset(X_train_tensor, y_train_tensor)

val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size

# Generator ensures reproducible split
train_subset, val_subset = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

# 2. Create DataLoaders
# Using the batch_size defined previously
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

print(f"Training samples: {len(train_subset)}")
print(f"Validation samples: {len(val_subset)}")

In [ ]:
def train_with_validation(model, optimizer, criterion, train_loader, val_loader, num_epochs, use_hms=False):
    hms_handler = HMSHandler(model) if use_hms else None
    history = {'loss': [], 'val_loss': []}
    
    for epoch in range(num_epochs):
        # --- Training Phase ---
        model.train()
        running_loss = 0.0
        for inputs, targets in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
        
        epoch_loss = running_loss / len(train_loader.dataset)
        history['loss'].append(epoch_loss)
        
        # --- Validation Phase ---
        model.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in val_loader:
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                running_val_loss += loss.item() * inputs.size(0)
                
        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        history['val_loss'].append(epoch_val_loss)

        # --- HMS Update ---
        if use_hms:
            hms_handler.on_epoch_end(epoch)
            
    return history

In [ ]:
# Capture initial weights for fair comparison
master_model, _, _ = get_model_and_optimizer('SGD')
initial_state = copy.deepcopy(master_model.state_dict())

# ==========================================
# 1. Generic Model (Validation Run)
# ==========================================
print("Training Generic Model with Validation...")
model_generic_val, optimizer_generic, criterion = get_model_and_optimizer('SGD')
model_generic_val.load_state_dict(copy.deepcopy(initial_state))

history_generic = train_with_validation(
    model_generic_val, 
    optimizer_generic, 
    criterion, 
    train_loader, 
    val_loader, 
    num_epochs, 
    use_hms=False
)

# Plot Generic
plt.figure(figsize=(6,4))
plt.plot(history_generic['loss'], label='loss')
plt.plot(history_generic['val_loss'], label='val_loss')
plt.title('Generic optimizer Loss')
plt.yscale('log')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper right')
plt.show()

In [ ]:
# ==========================================
# 2. HMS Model (Validation Run)
# ==========================================
print("Training HMS Model with Validation...")
model_hms_val, optimizer_hms, criterion = get_model_and_optimizer('SGD')
model_hms_val.load_state_dict(copy.deepcopy(initial_state))

history_hms = train_with_validation(
    model_hms_val, 
    optimizer_hms, 
    criterion, 
    train_loader, 
    val_loader, 
    num_epochs, 
    use_hms=True
)

# Plot HMS
plt.figure(figsize=(6,4))
plt.plot(history_hms['loss'], label='loss')
plt.plot(history_hms['val_loss'], label='val_loss')
plt.title('HMS-based optimizer Loss')
plt.yscale('log')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper right')
plt.show()

#### HMS update

In [ ]:
class HMSHandler:
    def __init__(self, model, r=1, t=100):
        self.model = model
        self.r = r
        self.t = t
        # Store initial weights (detached from graph)
        self.previous_weights = [p.data.clone() for p in model.parameters()]

    def on_epoch_end(self, epoch):
        # Update r factor every t epochs
        if (epoch + 1) % self.t == 0:
            self.r = round(self.r * 0.9, 4)

        with torch.no_grad(): # Critical: Do not track gradients for this manipulation
            for i, param in enumerate(self.model.parameters()):
                v1 = self.previous_weights[i] # Previous epoch weights
                v2 = param.data               # Current weights after optimizer step
                
                # --- Vectorized Apply HM Logic ---
                # Calculate magnitudes
                t1 = v1.abs()
                t2 = v2.abs()
                
                # Avoid division by zero for HM
                denom = t1 + t2
                # Mask where denominator is 0 or inputs are 0 (equivalent to v1==0 or v2==0)
                zero_mask = (v1 == 0) | (v2 == 0) | (denom == 0)
                
                # Harmonic Mean Calculation
                hm = torch.zeros_like(v1)
                hm[~zero_mask] = (2 * t1[~zero_mask] * t2[~zero_mask]) / denom[~zero_mask]
                
                # HMS Factor Calculation
                min1 = torch.min(t1, t2)
                hms_factor = (hm - min1).abs() * self.r
                
                # Update Logic
                # If v1 > v2 -> v2 = v2 - hms
                # If v1 < v2 -> v2 = v2 + hms
                diff_mask_pos = (v1 > v2)
                diff_mask_neg = (v1 < v2)
                
                # Apply updates
                v2[diff_mask_pos] = v2[diff_mask_pos] - hms_factor[diff_mask_pos]
                v2[diff_mask_neg] = v2[diff_mask_neg] + hms_factor[diff_mask_neg]
                
                # Update previous weights for next epoch
                self.previous_weights[i] = v2.clone()

In [ ]:
# per epoch basis
def train_model(model, optimizer, criterion, train_loader, num_epochs, log_file, use_hms=False):
    # Initialize HMS Handler if enabled
    hms_handler = HMSHandler(model) if use_hms else None
    
    # Store metrics for CSV logging
    history = []
    
    start_time = time.time()
    
    model.train()
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        
        for inputs, targets in train_loader:
            # 1. Forward Pass
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            # 2. Backward Pass & Optimization
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item() * inputs.size(0)
            
        # Average loss for the epoch
        epoch_loss /= len(train_loader.dataset)
        
        # 3. Apply HMS Update (Only if enabled)
        if use_hms:
            hms_handler.on_epoch_end(epoch)
            
        # Log metrics (mimicking CSVLogger)
        history.append({'epoch': epoch, 'loss': epoch_loss})
        
    end_time = time.time()
    
    # Save logs to CSV
    pd.DataFrame(history).to_csv(log_file, index=False)
    
    execution_time = round(end_time - start_time, 4)
    return execution_time

In [ ]:
# per epoch basis
# ==========================================
# 1. Train Generic Model (Without HMS)
# ==========================================
print("Training Generic Model...")
model_generic, optimizer_generic, criterion = get_model_and_optimizer(optimizer_name)
model_generic.load_state_dict(copy.deepcopy(initial_state)) # Load same starting weights

elapsed_training_time_generic = train_model(
    model_generic, 
    optimizer_generic, 
    criterion, 
    train_loader, 
    num_epochs, 
    generic_file, 
    use_hms=False
)
print('Execution time (Generic):', elapsed_training_time_generic, 'seconds')

# ==========================================
# 2. Train HMS Model (With HMS)
# ==========================================
print("\nTraining HMS Model...")
model_hms, optimizer_hms, criterion = get_model_and_optimizer(optimizer_name)
model_hms.load_state_dict(copy.deepcopy(initial_state)) # Load same starting weights

elapsed_training_time_hms = train_model(
    model_hms, 
    optimizer_hms, 
    criterion, 
    train_loader, 
    num_epochs, 
    hms_file, 
    use_hms=True
)
print('Execution time (HMS):', elapsed_training_time_hms, 'seconds')